# Win-weighted gate-head BC at scale (Kaggle #30 / vkhydras regime)

Tier-1 of the scale-up plan: **scaled supervised gate-head BC** on the full top-ladder
replay corpus, all persisted to Google Drive. Target: A100/H100, **< 5-6 h**.

Pipeline (each cell writes to Drive so a session reset resumes):
1. **Setup** — mount Drive, clone/pull repo, install deps
2. **Kaggle creds** — `kaggle.json` (pre-staged in Drive at `MyDrive/orbit_wars/kaggle.json`)
3. **Harvest** — `harvest_all.py`: all daily episode datasets, **newest-first**, capped (~47k games)
4. **Build shards** — `build_shards.py`: win-weighted, reduced-precision compressed shards
5. **Train** — `winbc_train.py --shards --gate_head`: scaled OrbitNet, learning curves to CSV
6. **Eval + curves** — arena win vs random / enders_1000 / reinforce_958 / producer; plot curves

The decoupled **launch/no-launch gate** (vkhydras's #1 fix) replaces hold-as-a-class — the
cure for the state-dependent no-op collapse that sank our prior BC. RL self-play (lagged
vs own checkpoints, NOT frozen-teacher) is the **Lambda GB100** follow-on, gated on these curves.

In [ ]:
# === Cell 1: Setup — mount Drive, clone/pull, install ===
import os, sys
from google.colab import drive
drive.mount('/content/drive')

DRIVE = '/content/drive/MyDrive/orbit_wars'   # Drive root: durable artifacts only
SHARDS  = f'{DRIVE}/shards'                     # built shards (persisted for resume)
CKPTS   = f'{DRIVE}/checkpoints'               # trained checkpoints + curves
# Replays live on LOCAL /content SSD, NOT Drive: harvest writes thousands of small files and
# the Drive mount is ~20x slower at that (it was the ~24-min/day bottleneck). Replays are
# disposable (only the shards need to persist) — if the session drops, re-harvest is fast.
REPLAYS = '/content/replays'
for d in (DRIVE, SHARDS, CKPTS, REPLAYS):
    os.makedirs(d, exist_ok=True)

REPO_URL = 'https://github.com/oshbocker/orbit_wars.git'  # <-- EDIT if needed
REPO_DIR = '/content/orbit_wars'
if os.path.exists(REPO_DIR):
    os.system(f'cd {REPO_DIR} && git pull')
else:
    os.system(f'git clone {REPO_URL} {REPO_DIR}')
os.chdir(REPO_DIR)
sys.path.insert(0, REPO_DIR)
os.system('pip install -q --upgrade "kaggle-environments>=1.28.0" "kaggle>=1.6" torch pyyaml')
import torch
print('CUDA:', torch.cuda.is_available(), torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU')
# Local disk sets the safe game cap (replays ~3 MB each): ~78 GB standard, more on A100.
print('--- local /content disk (sets MAX_TOTAL ceiling) ---')
os.system('df -h /content | tail -1')

In [ ]:
# === Cell 2: Kaggle credentials ===
# kaggle.json is already staged in Drive at f'{DRIVE}/kaggle.json' (uploaded via rclone).
# Just copy it into place. Fallback to interactive upload only if it's ever missing.
import os, shutil
os.makedirs(os.path.expanduser('~/.kaggle'), exist_ok=True)
src = f'{DRIVE}/kaggle.json'
dst = os.path.expanduser('~/.kaggle/kaggle.json')
if os.path.exists(src):
    shutil.copy(src, dst)
    print(f'kaggle.json loaded from {src}')
else:
    print(f'WARNING: {src} not found - uploading interactively (will be cached to Drive).')
    from google.colab import files
    files.upload()  # choose kaggle.json
    shutil.move('kaggle.json', dst)
    shutil.copy(dst, src)  # cache to Drive for next time
os.chmod(dst, 0o600)
# Sanity check: confirm the API authenticates before the harvest cell needs it.
from kaggle import KaggleApi
KaggleApi().authenticate()
print('kaggle.json ready + API authenticated')

In [ ]:
# === Cell 2.5: Replay guard — make re-runs disconnect-proof regardless of how the VM came back ===
# Replays live on LOCAL /content (fast SSD) by design, but /content is WIPED when Colab
# recycles the VM. This cell reconciles local vs a durable Drive mirror so a re-run "just
# works": it reuses whatever survived and only re-harvests what's genuinely missing.
#
#   - local has files  -> same runtime survived; keep them (Cell 3 will top-up/skip).
#   - local empty, Drive mirror has files -> VM was recycled; STAGE Drive -> local (no re-harvest).
#   - neither has files -> fresh start; Cell 3 harvests from scratch.
#
# Set MIRROR_TO_DRIVE=True to also push local replays UP to the Drive mirror at the end. Off by
# default: mirroring thousands of tiny files to the slow Drive mount is exactly the bottleneck
# the local-SSD design avoids. The durable resume target is the SHARDS (Cell 4), not replays —
# this mirror is only a convenience for skipping a re-harvest after a disconnect.
import os, glob

REPLAYS       = '/content/replays'            # local SSD — what Cell 3/4 read & write
REPLAYS_DRIVE = f'{DRIVE}/replays'            # durable mirror (survives VM recycle)
MIRROR_TO_DRIVE = False                       # flip True to persist local -> Drive after harvest
os.makedirs(REPLAYS, exist_ok=True)
os.makedirs(REPLAYS_DRIVE, exist_ok=True)

def _count_json(root):
    return sum(1 for _ in glob.iglob(os.path.join(root, '**', '*.json'), recursive=True))

n_local = _count_json(REPLAYS)
n_drive = _count_json(REPLAYS_DRIVE)
print(f'replays  local(/content): {n_local:>7}   drive(mirror): {n_drive:>7}')

if n_local == 0 and n_drive > 0:
    print(f'-> local empty, Drive mirror has {n_drive} games: staging Drive -> local (skips re-harvest)')
    os.system(f'cp -r {REPLAYS_DRIVE}/. {REPLAYS}/')
    n_local = _count_json(REPLAYS)
    print(f'-> staged. local now: {n_local}')
elif n_local > 0:
    print('-> local replays present (runtime survived or already staged); Cell 3 will top-up/skip.')
else:
    print('-> no replays anywhere yet; Cell 3 will harvest from scratch.')

if MIRROR_TO_DRIVE and n_local > 0:
    print(f'-> mirroring {n_local} local games UP to Drive (slow; one-time durability) ...')
    os.system(f'cp -r {REPLAYS}/. {REPLAYS_DRIVE}/')
    print(f'-> Drive mirror now: {_count_json(REPLAYS_DRIVE)}')

In [ ]:
# === Cell 3: Harvest replays to LOCAL disk (newest-first, fast, resumable) ===
# Writes to /content (fast SSD), NOT the Drive mount. ~4600 episodes/day available; newest
# days first so the cap keeps the most leaderboard-relevant data. Re-run to top up (skips
# files already on local disk). --timeout guards against a stalled download wedging a day.
#
# MAX_TOTAL is bounded by LOCAL DISK (~3 MB/game): 18k games ~= 54 GB (safe on ~78 GB standard
# Colab). Bump toward 47k only if Cell 1's `df -h` shows the headroom. This first pass is
# plenty to prove the regime; scale the full corpus on the Lambda tier.
MAX_TOTAL = 18000
PER_DAY   = 2000
WORKERS   = 16
!python scripts/harvest_all.py --start 2026-05-19 --per-day {PER_DAY} \
    --max-total {MAX_TOTAL} --workers {WORKERS} --timeout 60 --cache-root {REPLAYS}

In [ ]:
# === Cell 4: Build win-weighted sharded dataset (uses Colab's CPUs) ===
# CRITICAL for the time budget: training re-reads every shard each epoch. The Drive MOUNT is
# slow (~30 MB/s -> ~11 h of pure I/O over 30 epochs!), so we keep shards on LOCAL /content
# SSD (~400 MB/s -> ~2 min/epoch) and persist a copy to Drive only for resume-after-disconnect.
import os, json
SHARDS_LOCAL  = '/content/shards'          # fast local SSD — training reads from here
BUILD_WORKERS = 8
SHARD_SIZE    = 200000
os.makedirs(SHARDS_LOCAL, exist_ok=True)

if os.path.exists(f'{SHARDS}/manifest.json'):
    print('shards already on Drive -> staging to local /content for fast training')
    os.system(f'cp -r {SHARDS}/. {SHARDS_LOCAL}/')
else:
    # Build to local SSD (fast), then mirror to Drive so a disconnect after the build resumes.
    os.system(f'python scripts/build_shards.py --cache-root {REPLAYS} --out {SHARDS_LOCAL} '
              f'--shard-size {SHARD_SIZE} --workers {BUILD_WORKERS}')
    os.makedirs(SHARDS, exist_ok=True)
    os.system(f'cp -r {SHARDS_LOCAL}/. {SHARDS}/')   # one-time persist (~15-20 GB compressed)
print(json.load(open(f'{SHARDS_LOCAL}/manifest.json'))['n_examples'], 'examples (local)')

In [ ]:
# === Cell 5: Train scaled gate-head BC (streaming from LOCAL shards, curves to Drive) ===
# Scale capacity to data: ~2.7M (embed256/L4) for the first A100 pass; bump to embed512/L6
# (~15M) only once the corpus is large (vkhydras: big nets on few replays just overfit) and
# better suited to the Lambda tier. Reads from SHARDS_LOCAL (fast); ckpt+curves -> Drive.
# Best ckpt selected on gate-F1 + pointer-MRR (NOT top-1 launch_acc); .last.pt survives resets.
# --resume: if the Colab session drops mid-train, just RE-RUN this cell — it picks up from
# <out>.last.pt (model+optimizer+epoch) on Drive. First run (no ckpt) starts fresh from epoch 0.
EMBED, LAYERS, FF, HEADS = 256, 4, 512, 8
EPOCHS, BATCH, LR = 30, 1024, 3e-4
OUT    = f'{CKPTS}/winbc_gate.pt'
CURVES = f'{CKPTS}/winbc_gate_curves.csv'
!python scripts/winbc_train.py --shards {SHARDS_LOCAL} --gate_head --resume \
    --embed-dim {EMBED} --n-layers {LAYERS} --ff-dim {FF} --n-heads {HEADS} \
    --epochs {EPOCHS} --batch {BATCH} --lr {LR} \
    --launch_weight 8.0 --w_loss 0.3 \
    --out {OUT} --curves {CURVES}

In [ ]:
# === Cell 6a: Plot learning curves ===
import pandas as pd, matplotlib.pyplot as plt
df = pd.read_csv(CURVES)
fig, ax = plt.subplots(1, 2, figsize=(13, 4))
ax[0].plot(df.epoch, df.loss); ax[0].set_title('loss'); ax[0].set_xlabel('epoch')
for c in ['gate_f1', 'recall@5', 'mrr', 'gate_rec']:
    ax[1].plot(df.epoch, df[c], label=c)
ax[1].legend(); ax[1].set_title('gate / pointer quality'); ax[1].set_xlabel('epoch')
plt.tight_layout(); plt.show()
df.tail()

In [ ]:
# === Cell 6b: THE GATE — arena win vs the bots our prior BC lost to + producer ===
# Pre-registered bar (Kaggle #30 regime-check): the clone must MOVE under pressure and
# CRUSH the weak baselines; challenging producer is the real-signal stretch goal.
import os
os.environ['OW_BC_TEACHER_CKPT'] = OUT
# Movement-under-pressure (contested vs quiet launches/turn) + win rate:
for opp in ['random', 'enders_1000', 'reinforce_958', 'producer']:
    print(f'\n===== vs {opp} =====')
    os.system(f'python scripts/winbc_probe.py --opponent {opp} --games 12')
# Paired arena (THE gate metric) for a high-n read on the strong bots:
!python scripts/arena.py --agents bc_teacher,enders_1000,reinforce_958,producer \
    --games 60 --workers 4

## Reading the result → next tier

- **Moves under pressure (contested l/t high) + beats the weak bots + curves still rising**
  → the regime works. Scale up (more data via Cell 3 `MAX_TOTAL`, bigger net via Cell 5
  `EMBED/LAYERS`), then take **BC-init + lagged self-play PPO** to Lambda GB100 (~10 h).
  Self-play opponents = our **own earlier checkpoints**, +/-1 reward — NOT a frozen teacher
  / KL anchor (that stalls at teacher level; our documented ExIt mistake).
- **Still passive under pressure / loses to weak bots** → diagnose before GPU spend:
  executor sizing (`OW_BC_DRAIN=full` ablation), gate threshold (`OW_BC_GATE_THR`),
  or data balance — not more parameters.

Download the best checkpoint locally with `scripts/download_checkpoint.py` (or rclone from
`{DRIVE}/checkpoints`) and re-verify with `scripts/eval_fast.py` / `scripts/arena.py`.

In [ ]:
# === Heartbeat — run ANYTIME after a reconnect to see if the pipeline is alive & which phase ===
# Self-contained (doesn't need earlier cells to have run this session): recomputes paths from
# DRIVE if defined, else the notebook defaults. Re-run it every minute or two — the numbers
# that should be MOVING tell you it's alive:
#   harvest phase -> replay json count climbs       build phase -> shard count climbs
#   train phase   -> curves.csv row count climbs (THE durable heartbeat; on Drive)
import os, glob, subprocess

DRIVE   = globals().get('DRIVE', '/content/drive/MyDrive/orbit_wars')
CKPTS   = globals().get('CKPTS', f'{DRIVE}/checkpoints')
CURVES  = globals().get('CURVES', f'{CKPTS}/winbc_gate_curves.csv')
REPLAYS = globals().get('REPLAYS', '/content/replays')
SHARDS_LOCAL = globals().get('SHARDS_LOCAL', '/content/shards')

def _sh(cmd):
    return subprocess.run(cmd, shell=True, capture_output=True, text=True).stdout.strip()

def _count_json(root):
    return sum(1 for _ in glob.iglob(os.path.join(root, '**', '*.json'), recursive=True))

# 1. Which phase is actually running? (empty => no pipeline proc alive)
procs = _sh("ps -eo etimes,pid,cmd | grep -E 'harvest_all|build_shards|winbc_train' | grep -v grep")
print('=== running pipeline process (elapsed_s  pid  cmd) ===')
print(procs if procs else '(none — cell finished, died on disconnect, or not started)')

# 2. GPU: busy => training; idle => harvest/build (CPU/network) or nothing running.
print('\n=== GPU ===')
print(_sh("nvidia-smi --query-gpu=utilization.gpu,memory.used,memory.total "
          "--format=csv,noheader 2>/dev/null") or '(no GPU / nvidia-smi unavailable)')

# 3. Progress counters — run this cell twice ~30s apart; the active phase's number moves.
print('\n=== progress counters ===')
print(f'replays (local) : {_count_json(REPLAYS):>7}   {REPLAYS}')
n_shard = len(glob.glob(os.path.join(SHARDS_LOCAL, '*.npz')))   # build_shards writes shard_w*_*.npz
manifest = os.path.join(SHARDS_LOCAL, 'manifest.json')
print(f'shards  (local) : {n_shard:>7} files   manifest={"yes" if os.path.exists(manifest) else "no"}')
print(f'disk /content   : ' + (_sh("df -h /content | tail -1") or '?'))

# 4. Training heartbeat — curve rows + last line (this file is on Drive => survives VM recycle).
print('\n=== training curves (Drive — the durable heartbeat) ===')
if os.path.exists(CURVES):
    rows = _sh(f"wc -l < '{CURVES}'")
    mtime = _sh(f"date -r '{CURVES}' '+%Y-%m-%d %H:%M:%S'")
    print(f'{CURVES}\n  rows={rows}  last_modified={mtime}')
    print('  head:', _sh(f"head -1 '{CURVES}'"))
    print('  tail:', _sh(f"tail -1 '{CURVES}'"))
else:
    print(f'{CURVES}\n  (not created yet — training has not reached its first epoch checkpoint)')